In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
nvidia = pd.read_csv('nvidia_cleaned.csv')

### Relative Strength Index (RSI)

In [3]:
delta = nvidia['Close'].diff()

gain = delta.where(delta > 0, 0)
loss = -delta.where(delta < 0, 0)

avg_gain = gain.ewm(alpha=1/14, min_periods=14).mean()
avg_loss = loss.ewm(alpha=1/14, min_periods=14).mean()

rs = avg_gain / avg_loss

nvidia['RSI'] = 100 - (100 / (1 + rs))

In [4]:
nvidia['RSI'].describe()

count    6783.000000
mean       53.689646
std        12.799474
min        13.025447
25%        44.545816
50%        53.898414
75%        62.809986
max        91.420719
Name: RSI, dtype: float64

### MACD

In [5]:
ema12 = nvidia['Close'].ewm(span=12, adjust=False).mean()
ema26 = nvidia['Close'].ewm(span=26, adjust=False).mean()

nvidia['MACD'] = ema12 - ema26

nvidia['Signal_Line'] = nvidia['MACD'].ewm(span=9, adjust=False).mean()

In [6]:
nvidia['MACD'], nvidia['Signal_Line']

(0       0.000000
 1       0.000314
 2       0.000300
 3       0.000275
 4       0.000244
           ...   
 6791   -0.118336
 6792    0.144606
 6793    0.253241
 6794    0.499000
 6795    0.925177
 Name: MACD, Length: 6796, dtype: float64,
 0       0.000000
 1       0.000063
 2       0.000110
 3       0.000143
 4       0.000163
           ...   
 6791    0.309547
 6792    0.276559
 6793    0.271895
 6794    0.317316
 6795    0.438889
 Name: Signal_Line, Length: 6796, dtype: float64)

-MACD crosses above signal line - Buy Signal 

-MACD crosses below signal line - Sell Signal

In [7]:
nvidia['MACD_Histogram'] = (
    nvidia['MACD'] - nvidia['Signal_Line']
)

### Bollinger Bands

In [8]:
rolling_std = nvidia['Close'].rolling(window=20).std()

nvidia['Upper_Band'] = nvidia['MA20'] + (rolling_std * 2)
nvidia['Lower_Band'] = nvidia['MA20'] - (rolling_std * 2)

### Momentum Indicator

In [9]:
nvidia['Momentum'] = (
    (nvidia['Close'] / nvidia['Close'].shift(10)) - 1
) * 100

In [10]:
nvidia['Momentum']

0            NaN
1            NaN
2            NaN
3            NaN
4            NaN
          ...   
6791   -2.257947
6792    1.421317
6793    0.870930
6794    1.935764
6795    3.073035
Name: Momentum, Length: 6796, dtype: float64

### LAG Features

In [11]:
nvidia['Lag_1'] = nvidia['Close'].shift(1)
nvidia['Lag_2'] = nvidia['Close'].shift(2)
nvidia['Lag_3'] = nvidia['Close'].shift(3)
nvidia['Lag_5'] = nvidia['Close'].shift(5)

### Volume Features

In [12]:
nvidia['Volume_Change'] = nvidia['Volume'].pct_change()

nvidia['Volume_MA20'] = nvidia['Volume'].rolling(20).mean()

In [16]:
nvidia.to_csv('nvidia_with_features.csv', index=False)